# PCA Algorithms: Interactive Exploration

## 🎯 Learning Objectives

By the end of this notebook, you will:
1. Understand **three different algorithms** for computing PCA
2. Visualize **step-by-step** how NIPALS iteratively finds principal components
3. Compare **eigenvalue decomposition** vs **iterative** approaches
4. See how all methods produce the **same results**

---

## 📖 Algorithms Covered

| Algorithm | Approach | Advantages |
|-----------|----------|------------|
| **NIPALS** | Iterative regression | Handles missing data, memory efficient |
| **Eigenvalue Decomposition** | Matrix factorization | Direct solution, mathematically elegant |
| **SVD (sklearn)** | Singular value decomposition | Numerically stable, standard library |

---

# Test data

In [ ]:
import numpy as np
from scipy import linalg as LA
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, VBox, HBox, Output, HTML
from IPython.display import display, clear_output
from sklearn.decomposition import PCA as sklearn_PCA

np.random.seed(42)

# Original planet data
X = np.array([
        [0.387,4878, 5.42],   # Mercury
        [0.723,12104,5.25],   # Venus
        [1,12756,5.52],       # Earth
        [1.524,6787,3.94],    # Mars
    ])

print("✓ Libraries loaded successfully!")
print("\nOriginal planet data shape:", X.shape)

✓ Libraries loaded successfully!

Original planet data shape: (4, 3)


---

In [2]:
def generate_data(n_samples=100, var_x=5.0, var_y=1.0, angle=30):
    """
    Generate 2D Gaussian data with controllable variance and rotation.
    
    Parameters:
    -----------
    n_samples : int
        Number of data points
    var_x, var_y : float
        Variances along principal axes
    angle : float
        Rotation angle in degrees
        
    Returns:
    --------
    X : ndarray of shape (n_samples, 2)
        Generated data (centered)
    """
    # Generate data in canonical orientation
    u = np.sqrt(var_x) * np.random.randn(n_samples)
    v = np.sqrt(var_y) * np.random.randn(n_samples)
    data = np.vstack([u, v])
    
    # Rotate
    theta = np.deg2rad(angle)
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    X = (R @ data).T
    
    # Center the data
    X = X - np.mean(X, axis=0)
    
    return X


def plot_data_simple(X, title="Generated Data"):
    """Plot 2D data."""
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=X[:, 0], y=X[:, 1],
        mode='markers',
        marker=dict(size=8, color='blue', opacity=0.6),
        name='Data'
    ))
    
    L = np.max(np.abs(X)) * 1.3
    fig.update_layout(
        title=title,
        xaxis=dict(range=[-L, L], title='X₁', scaleanchor='y'),
        yaxis=dict(range=[-L, L], title='X₂'),
        width=600, height=500
    )
    fig.show()


# Interactive data generator
def interactive_data(n_samples=100, var_x=5.0, var_y=1.0, angle=30):
    X_2d = generate_data(n_samples, var_x, var_y, angle)
    plot_data_simple(X_2d, title=f"Generated Data (n={len(X_2d)}, angle={angle}°)")
    return X_2d

# Create widget
interact(
    interactive_data,
    n_samples=IntSlider(min=20, max=300, step=20, value=100, description='Samples'),
    var_x=FloatSlider(min=1, max=10, step=0.5, value=5, description='Var X'),
    var_y=FloatSlider(min=0.1, max=5, step=0.1, value=1, description='Var Y'),
    angle=IntSlider(min=0, max=90, step=5, value=30, description='Angle')
);

interactive(children=(IntSlider(value=100, description='Samples', max=300, min=20, step=20), FloatSlider(value…

---

# Section 1: Generate Interactive 2D Data

For better visualization, we'll first work with 2D synthetic data. You can control the data characteristics with sliders below:

# Non-Iinear Iterative Partial Least-Squares (NIPALS) algorithm

## ALGORITHM

Steps to compute PCA using NIPALS algorithm

* **Step 1:** Initialize an arbitrary column vector $\mathbf{t}_{a}$ either randomly or by just copying any column of X. 
* **Step 2:** Take every column of $X$, $X_{k}$ and regress it onto the $\mathbf{t}_{a}$ vector and store the regression coefficients as $\mathbf{p}_{ka}$. (Note: This simply means performing an ordinary least squares regression ($y=mx$) with $x=t_{a}$ and $y=X_{k}$ with $m=(\mathbf{x}^T\mathbf{x})^{-1}\mathbf{x}^T\mathbf{y}$). In the current notation we get 

$p_{ka}=\frac{\mathbf{t}_{a}^T\mathbf{X}_{k}}{\mathbf{t}_{a}^T\mathbf{t}_{a}}$

Repeat it for each of the columns of $X$ to get the entire vector $\mathbf{p}_{k}$. This is shown in the illustration above where each column from $X$ is regressed, one at a time, on $\mathbf{t}_{a}$, to calculate the loading entry, $\mathbf{p}_{ka}$. In practice we don't do this one column at a time; we can regress all columns in $X$ in one go: 

$\mathbf{p}_{a}^T=\frac{1}{\mathbf{t}_{a}^T\mathbf{t}_{a}}\mathbf{t}_{a}^T\mathbf{X}_{a}$ 

where $\mathbf{t}_{a}$ is an $N \times 1$ column vector, and $\mathbf{X}_{a}$ is an $N \times K$ matrix.
* **Step 3:** The loading vector $\mathbf{p}_{a}^T$ won't have unit length (magnitude) yet. So we simply rescale it to have magnitude of 1.0: 

$\mathbf{p}_{a}=\frac{\mathbf{p}_{a}}{\sqrt{\mathbf{p}_{a}^T\mathbf{p}_{a}}}$

* **Step 4:** Regress every row in $X$ onto this normalized loadings vector. As illustrated below, in our linear regression the rows in X are our y-variable each time, while the loadings vector is our x-variable. The regression coefficient becomes the score value for that $i^{th}$ row:

$t_{i,a}=\frac{\mathbf{x}_{i}^T\mathbf{p}_{a}}{\mathbf{p}_{a}^T\mathbf{p}_{a}}$

where $\mathbf{x}_{i}^T$ is a $K \times 1$ column vector. We can combine these $N$ separate least-squares models and calculate them in one go to get the entire vector: 

$\mathbf{t}_{a}=\frac{\mathbf{X}\mathbf{p}_{a}}{\mathbf{p}_{a}^T\mathbf{p}_{a}}$

where $\mathbf{p}_{a}$ is a $K \times 1$ column vector.
* **Step 5:** Continue looping over steps 2, 3, 4 until the change in vector $\mathbf{t}_{a}$ is below a chosen tolerance
* **Step 6:** On convergence, the score vector and the loading vectors, $\mathbf{t}_{a}$ and $\mathbf{p}_{a}$ are stored as the $a^{th}$ column in matrix $\mathbf{T}$ and $\mathbf{P}$. We then deflate the $\mathbf{X}$ matrix. This crucial step removes the variability captured in this component ($\mathbf{t}_{a}$ and $\mathbf{p}_{a}$) from $\mathbf{X}$:

$E_{a}=X_{a}-\mathbf{t}_{a}\mathbf{p}_{a}^T$

$X_{a+1} = E_{a}$ 

For the first component, $X_{a}$ is just the preprocessed raw data. So we can see that the second component is actually calculated on the residuals $E_{1}$, obtained after extracting the first component. This is called deflation, and nicely shows why each component is orthogonal to the others. Each subsequent component is only seeing variation remaining after removing all the others; there is no possibility that two components can explain the same type of variability. After deflation we go back to step 1 and repeat the entire process for the next component.

## IMPLEMENTATION IN PYTHON

In [ ]:
def nipals_step_by_step(X, max_iter=20, tol=1e-6):
    """
    NIPALS algorithm with step-by-step history tracking.
    
    Parameters:
    -----------
    X : ndarray
        Data matrix (will be centered)
    max_iter : int
        Maximum iterations
    tol : float
        Convergence tolerance
        
    Returns:
    --------
    history : list of dicts
        Each dict contains: iteration, p (loading), t (score), change
    """
    history = []
    
    # Center data
    X_centered = X - np.mean(X, axis=0)
    n, k = X_centered.shape
    
    # Initialize: use first column as starting scores
    t = X_centered[:, 0].reshape(-1, 1)
    
    for iteration in range(max_iter):
        # Step 2: Regress each X column onto t to get loadings
        p = (X_centered.T @ t) / (t.T @ t)
        
        # Step 3: Normalize loading vector
        p = p / np.linalg.norm(p)
        
        # Step 4: Regress each X row onto p to get new scores
        t_new = (X_centered @ p) / (p.T @ p)
        
        # Calculate change
        change = np.linalg.norm(t_new - t)
        
        # Store history
        history.append({
            'iteration': iteration + 1,
            'p': p.flatten(),
            't': t_new.flatten(),
            'change': change
        })
        
        # Check convergence
        if change < tol:
            break
            
        t = t_new
    
    return history

# Generate data for NIPALS demonstration
X_nipals = generate_data(100, var_x=5, var_y=1, angle=35)

# Run NIPALS and get history
history = nipals_step_by_step(X_nipals)

print(f"✓ NIPALS converged in {len(history)} iterations")
print(f"\nFinal loading vector: {history[-1]['p']}")
print(f"Final change: {history[-1]['change']:.2e}")

In [3]:
def plot_nipals_convergence():
    """Show how NIPALS converges over iterations."""
    iterations = [h['iteration'] for h in history]
    changes = [h['change'] for h in history]
    angles = [np.degrees(np.arctan2(h['p'][1], h['p'][0])) for h in history]
    
    fig = make_subplots(rows=1, cols=2, 
                        subplot_titles=('Convergence (Change in t)', 'Loading Angle vs Iteration'))
    
    # Change over iterations (log scale)
    fig.add_trace(
        go.Scatter(x=iterations, y=changes, mode='lines+markers',
                  marker=dict(size=10), line=dict(width=2)),
        row=1, col=1
    )
    fig.update_yaxes(type='log', title='Change (log scale)', row=1, col=1)
    fig.update_xaxes(title='Iteration', row=1, col=1)
    
    # Angle over iterations
    fig.add_trace(
        go.Scatter(x=iterations, y=angles, mode='lines+markers',
                  marker=dict(size=10, color='red'), line=dict(width=2, color='red')),
        row=1, col=2
    )
    fig.add_hline(y=35, line_dash='dash', line_color='green', 
                  annotation_text='True angle', row=1, col=2)
    fig.update_yaxes(title='PC1 Angle (degrees)', row=1, col=2)
    fig.update_xaxes(title='Iteration', row=1, col=2)
    
    fig.update_layout(title='NIPALS Convergence Analysis', width=1000, height=400, showlegend=False)
    fig.show()

plot_nipals_convergence()

NameError: name 'history' is not defined

In [ ]:
def show_nipals_iteration(iteration=1):
    """Visualize NIPALS at a specific iteration."""
    idx = min(iteration - 1, len(history) - 1)
    state = history[idx]
    p = state['p']
    
    fig = go.Figure()
    
    # Data points
    fig.add_trace(go.Scatter(
        x=X_nipals[:, 0], y=X_nipals[:, 1],
        mode='markers',
        marker=dict(size=8, color='blue', opacity=0.5),
        name='Data'
    ))
    
    # Current loading vector
    L = np.max(np.abs(X_nipals)) * 0.9
    fig.add_trace(go.Scatter(
        x=[-p[0]*L, p[0]*L], y=[-p[1]*L, p[1]*L],
        mode='lines',
        line=dict(color='red', width=4),
        name=f'PC1 (iter {state["iteration"]})'
    ))
    
    # Show projections onto current PC
    proj_coords = X_nipals @ p
    proj_points = np.outer(proj_coords, p)
    
    # Sample projection lines (every 10th point)
    for i in range(0, len(X_nipals), 10):
        fig.add_trace(go.Scatter(
            x=[X_nipals[i, 0], proj_points[i, 0]],
            y=[X_nipals[i, 1], proj_points[i, 1]],
            mode='lines',
            line=dict(color='gray', width=1, dash='dash'),
            showlegend=False
        ))
    
    # Projected points
    fig.add_trace(go.Scatter(
        x=proj_points[:, 0], y=proj_points[:, 1],
        mode='markers',
        marker=dict(size=5, color='red', opacity=0.6),
        name='Projections'
    ))
    
    L = np.max(np.abs(X_nipals)) * 1.2
    fig.update_layout(
        title=f"NIPALS Iteration {state['iteration']} | Change: {state['change']:.2e}",
        xaxis=dict(range=[-L, L], title='X₁', scaleanchor='y'),
        yaxis=dict(range=[-L, L], title='X₂'),
        width=700, height=600
    )
    fig.show()
    
    # Show convergence info
    angle = np.degrees(np.arctan2(p[1], p[0]))
    print(f"Loading vector p = [{p[0]:.4f}, {p[1]:.4f}]")
    print(f"Angle: {angle:.1f}° (true data angle: 35°)")

# Create slider widget
interact(
    show_nipals_iteration,
    iteration=IntSlider(min=1, max=len(history), step=1, value=1, 
                        description='Iteration', continuous_update=False)
);

interactive(children=(IntSlider(value=1, continuous_update=False, description='Iteration', max=9, min=1), Outp…

In [ ]:
def nipals_step_by_step(X, max_iter=20, tol=1e-6):
    """
    NIPALS algorithm that returns intermediate states for visualization.
    
    Returns:
    --------
    history : list of dicts
        Each dict contains 't', 'p', 'change' for that iteration
    """
    n, k = X.shape
    
    # Initialize t with first column
    t = X[:, 0].copy().reshape(-1, 1)
    
    history = []
    
    for iteration in range(max_iter):
        # Step 2: Regress X onto t to get p
        p = (X.T @ t) / (t.T @ t)
        
        # Step 3: Normalize p
        p = p / np.linalg.norm(p)
        
        # Step 4: Regress X onto p to get new t
        t_new = (X @ p) / (p.T @ p)
        
        # Calculate change
        change = np.linalg.norm(t_new - t)
        
        # Store history
        history.append({
            'iteration': iteration + 1,
            't': t_new.flatten(),
            'p': p.flatten(),
            'change': change
        })
        
        t = t_new
        
        if change < tol:
            break
    
    return history


# Generate fixed data for NIPALS demo
X_nipals = generate_data(100, var_x=5, var_y=1, angle=35)

# Run NIPALS and get history
history = nipals_step_by_step(X_nipals)

print(f"✓ NIPALS converged in {len(history)} iterations")
print(f"\nFinal loading vector: {history[-1]['p']}")
print(f"Final change: {history[-1]['change']:.2e}")

✓ NIPALS converged in 9 iterations

Final loading vector: [0.81921454 0.57348718]
Final change: 5.22e-07


---

## 🎮 Interactive NIPALS Visualization

Watch how the loading vector (red arrow) iteratively converges to find PC1:

In [ ]:
def PCA_NIPALS(X, no_components):
    """
    PCA using NIPALS algorithm.
    
    Parameters:
    -----------
    X : ndarray
        Data matrix (will be centered)
    no_components : int
        Number of principal components to extract
        
    Returns:
    --------
    T : ndarray
        Score matrix
    P : ndarray
        Loading matrix
    pcvar : ndarray
        Variance explained by each component
    """
    tol = 0.0000001
    it = 1000
    obsCount, varCount = X.shape
    Xa = X - np.mean(X, axis=0) 
    
    T = np.zeros((obsCount, no_components))
    P = np.zeros((varCount, no_components))
    pcvar = np.zeros((varCount, 1))
    varTotal = np.sum(np.var(Xa, axis=0, ddof=1))
    currVar = varTotal
    nr = 0
    
    for h in range(no_components):
        th = Xa[:, 0].reshape(-1, 1)
        ende = False
        while ende != True:
            nr = nr + 1
            ph = np.dot(Xa.T, th) / np.dot(th.T, th)
            ph = ph / np.linalg.norm(ph)
            thnew = np.dot(Xa, ph) / np.dot(ph.T, ph)
            prec = np.dot((thnew - th).T, (thnew - th))
            th = thnew
            if prec <= (tol * tol):
                ende = True
            elif it <= nr:
                ende = True
                print("Iteration stops without convergence")
        Ea = Xa - np.dot(th, ph.T)
        Xa = Ea    
        T[:, h] = th.flatten()
        P[:, h] = ph.flatten()
        oldVar = currVar
        currVar = np.sum(np.var(Xa, axis=0, ddof=1))
        pcvar[h] = (oldVar - currVar) / varTotal
        nr = 0
    return T, P, pcvar

## Advantages of the NIPALS algorithm
* The NIPALS algorithm computes one component at a time. The first component computed is
equivalent to the t1 and p1 vectors that would have been found from an eigenvalue or singular value
decomposition.
* The algorithm can handle missing data in X.
* The algorithm always converges, but the convergence can sometimes be slow.
* It is also known as the Power algorithm to calculate eigenvectors and eigenvalues.
* It works well for very large data sets.
* It is used by most software packages, especially those that handle missing data.
* Of interest: it is well known that Google used this algorithm for the early versions of their search engine, called PageRank148.

In [ ]:
no_components = 3
T, P, pcvar = PCA_NIPALS(X, no_components)

In [ ]:
T

array([[-4.25324997e+03, -8.41288672e-01,  8.37859036e-03],
       [ 2.97275001e+03, -1.25977272e-01, -1.82476780e-01],
       [ 3.62475003e+03, -1.56843494e-01,  1.65224286e-01],
       [-2.34425007e+03,  1.12410944e+00,  8.87390330e-03]])

In [ ]:
P

array([[ 1.21901390e-05,  5.66460728e-01,  8.24088735e-01],
       [ 9.99999997e-01,  5.32639787e-05, -5.14047689e-05],
       [ 7.30130279e-05, -8.24088733e-01,  5.66460726e-01]])

In [ ]:
pcvar

array([[9.99999955e-01],
       [4.41567976e-08],
       [1.33326424e-09]])

# Eigenvalue decomposition approach

## ALGORITHM
Recall that the latent variable directions (the loading vectors) were oriented so that the variance of the
scores in that direction were maximal. We can cast this as an optimization problem. For the first
component: $$max (\phi)=\mathbf{t}_{1}^{'}\mathbf{t}_{1}=\mathbf{p}_{1}^{'} \mathbf{X}^{'}\mathbf{X}\mathbf{p}_{1}$$
such that $$\mathbf{p}_{1}^{'}\mathbf{p}_{1}=1$$.

This is equivalent to $$max (\phi)=\mathbf{p}_{1}^{'} \mathbf{X}^{'}\mathbf{X}\mathbf{p}_{1}-\lambda(\mathbf{p}_{1}^{'}\mathbf{p}_{1}-1)$$ because we can move the constraint into the objective function with a Lagrange multiplier, $\lambda$. The maximum value must occur when the partial derivatives with respect to $\mathbf{p}_{1}$, our search variable, are zero: $$\frac{\partial \phi}{\partial p_1}= \frac{\partial (\mathbf{p}_{1}^{'} \mathbf{X}^{'}\mathbf{X}\mathbf{p}_{1}-\lambda(\mathbf{p}_{1}^{'}\mathbf{p}_{1}-1))}{\partial \mathbf{p}_1}=0$$
$$2\mathbf{X}^{'}\mathbf{X}\mathbf{p}_{1}-2\lambda_{1}\mathbf{p}_{1}=0$$
$$(\mathbf{X}^{'}\mathbf{X}-\lambda_{1}\mathbf{I})\mathbf{p}_1=0$$
$$\mathbf{X}^{'}\mathbf{X}\mathbf{p}_1=\lambda_{1}\mathbf{p}_1$$

which is just the eigenvalue equation, indicating that $\mathbf{p}_1$ is the eigenvector of $\mathbf{X}^{'}\mathbf{X}$ and $\lambda_{1}$ is the eigenvalue. One can show that $\lambda_{1}=\mathbf{t}_{1}^{'}\mathbf{t}_{1}$, which is proportional to the variance of the first component. In a similar manner we can calculate the second eigenvalue, but this time we add the additional constraint that $\mathbf{p}_1 \perp \mathbf{p}_2$. Writing out this objective function and taking partial derivatives leads to showing that $$\mathbf{X}^{'}\mathbf{X}\mathbf{p}_2 = \lambda_2 \mathbf{p}_2$$.

From this we learn that:
* The loadings are the eigenvectors of $\mathbf{X}^{'}\mathbf{X}$.
* Sorting the eigenvalues in order from largest to smallest gives the order of the corresponding eigenvectors, the loadings.
* We know from the theory of eigenvalues that if there are distinct eigenvalues, then their eigenvectors are linearly independent (orthogonal).
* We also know the eigenvalues of $\mathbf{X}^{'}\mathbf{X}$ must be real values and positive; this matches with the interpretation that the eigenvalues are proportional to the variance of each score vector.
* Also, the sum of the eigenvalues must add up to sum of the diagonal entries of $\mathbf{X}^{'}\mathbf{X}$, which represents of the total variance of the X matrix, if all eigenvectors are extracted. So plotting the eigenvalues is equivalent to showing the proportion of variance explained in X by each component. This is not necessarily a good way to judge the number of components to use, but it is a rough guide: use a Pareto plot of the eigenvalues (though in the context of eigenvalue problems, this plot is called a scree plot).

### 🔍 Key Observations

1. **Eigenvector lengths** scale with square root of eigenvalues (variance along that direction)
2. **Off-diagonal** elements of covariance matrix show correlation between variables
3. **Eigenvalues** directly give the variance explained by each PC
4. When **angle = 0°**, covariance matrix is diagonal (uncorrelated variables)

<details>
<summary>💡 Try this experiment</summary>

Set `Var X = Var Y = 3` and watch how the eigenvalues become equal. What does this mean for PCA?

**Answer:** When variances are equal and there's no correlation, all directions are equally good. PCA becomes arbitrary!
</details>

---

In [ ]:
def PCA_Eigenvalue(X):
    """
    PCA using eigenvalue decomposition of the covariance matrix.
    
    Returns:
    --------
    scores : ndarray
        Projected data (T = X @ P)
    loadings : ndarray
        Eigenvectors (columns are PCs)
    eigenvalues : ndarray
        Eigenvalues (variance along each PC)
    cov_matrix : ndarray
        The covariance matrix (for visualization)
    """
    # Center the data
    X_centered = X - np.mean(X, axis=0)
    
    # Compute covariance matrix
    cov_matrix = np.cov(X_centered, rowvar=False)
    
    # Eigenvalue decomposition
    eigenvalues, eigenvectors = LA.eigh(cov_matrix)
    
    # Sort by eigenvalue (descending)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # Compute scores
    scores = X_centered @ eigenvectors
    
    return scores, eigenvectors, eigenvalues, cov_matrix


def visualize_eigenvalue_pca(var_x=5.0, var_y=1.0, angle=35):
    """Interactive visualization of eigenvalue-based PCA."""
    # Generate data
    X_vis = generate_data(150, var_x, var_y, angle)
    
    # Perform PCA
    scores, loadings, eigenvalues, cov = PCA_Eigenvalue(X_vis)
    
    # Create subplot figure
    fig = make_subplots(rows=1, cols=3, 
                        subplot_titles=('Data + PCs', 'Covariance Matrix', 'Eigenvalue Spectrum'),
                        specs=[[{'type': 'scatter'}, {'type': 'heatmap'}, {'type': 'bar'}]])
    
    # Panel 1: Data with eigenvectors
    fig.add_trace(
        go.Scatter(x=X_vis[:, 0], y=X_vis[:, 1], mode='markers',
                  marker=dict(size=6, color='blue', opacity=0.5), name='Data'),
        row=1, col=1
    )
    
    colors = ['red', 'green']
    for i in range(2):
        vec = loadings[:, i] * np.sqrt(eigenvalues[i]) * 2
        fig.add_trace(
            go.Scatter(x=[-vec[0], vec[0]], y=[-vec[1], vec[1]], mode='lines',
                      line=dict(color=colors[i], width=4), name=f'PC{i+1} (λ={eigenvalues[i]:.2f})'),
            row=1, col=1
        )
    
    # Panel 2: Covariance matrix heatmap
    fig.add_trace(
        go.Heatmap(z=cov, x=['X₁', 'X₂'], y=['X₁', 'X₂'],
                   colorscale='RdBu', zmid=0,
                   text=np.round(cov, 2), texttemplate='%{text}', showscale=False),
        row=1, col=2
    )
    
    # Panel 3: Eigenvalue bar chart
    total_var = np.sum(eigenvalues)
    fig.add_trace(
        go.Bar(x=['PC1', 'PC2'], y=eigenvalues, 
               marker=dict(color=['red', 'green']),
               text=[f'{e/total_var*100:.1f}%' for e in eigenvalues],
               textposition='outside', showlegend=False),
        row=1, col=3
    )
    
    fig.update_layout(width=1200, height=450, showlegend=True)
    fig.update_xaxes(scaleanchor='y', row=1, col=1)
    fig.update_yaxes(title='Eigenvalue', row=1, col=3)
    fig.show()
    
    print(f"Covariance matrix:")
    print(f"  Var(X₁) = {cov[0,0]:.3f}")
    print(f"  Var(X₂) = {cov[1,1]:.3f}")
    print(f"  Cov(X₁,X₂) = {cov[0,1]:.3f}")
    print(f"\nEigenvalues: [{eigenvalues[0]:.3f}, {eigenvalues[1]:.3f}]")
    print(f"Explained variance: [{eigenvalues[0]/total_var*100:.1f}%, {eigenvalues[1]/total_var*100:.1f}%]")

# Create interactive widget
interact(
    visualize_eigenvalue_pca,
    var_x=FloatSlider(min=1, max=10, step=0.5, value=5, description='Var X'),
    var_y=FloatSlider(min=0.5, max=5, step=0.5, value=1, description='Var Y'),
    angle=IntSlider(min=0, max=90, step=5, value=35, description='Angle')
);

interactive(children=(FloatSlider(value=5.0, description='Var X', max=10.0, min=1.0, step=0.5), FloatSlider(va…

---

## 🎮 Interactive Eigenvalue Visualization

Explore how data characteristics affect the covariance matrix and eigenvectors:

In [ ]:
cov = np.cov(X, rowvar = False)
evals , evecs = LA.eigh(cov)
idx = np.argsort(evals)[::-1]
evecs = evecs[:,idx]
evals = evals[idx]
score_eig = np.dot(X, evecs) 

In [ ]:
score_eig

array([[-4.87800039e+03,  3.98751895e+00,  3.13838701e+00],
       [-1.21040004e+04,  3.27220755e+00,  2.94753164e+00],
       [-1.27560004e+04,  3.30307377e+00,  3.29523270e+00],
       [-6.78700029e+03,  2.02212084e+00,  3.13888232e+00]])

In [ ]:
evecs

array([[-1.21901390e-05, -5.66460727e-01,  8.24088736e-01],
       [-9.99999997e-01, -5.32639789e-05, -5.14047691e-05],
       [-7.30130279e-05,  8.24088734e-01,  5.66460725e-01]])

In [ ]:
evals

array([1.51872330e+07, 6.70619604e-01, 2.02485956e-02])

# SKLEARN PCA

---

# Section 3: When to Use Which Algorithm?

## 📊 Algorithm Comparison

| Feature | NIPALS | Eigenvalue | SVD (sklearn) |
|---------|--------|------------|---------------|
| **Speed (small data)** | Slower | Fast | Fast |
| **Speed (large data)** | Efficient | Memory issues | Very efficient |
| **Missing data** | ✅ Handles it | ❌ Requires complete | ❌ Requires complete |
| **Partial extraction** | ✅ One PC at a time | ❌ All at once | ✅ Can specify k |
| **Numerical stability** | Good | Can be unstable | Very stable |
| **Interpretability** | Iterative (educational) | Direct (elegant) | Black box |

---

## 🎯 Recommendations

| Situation | Best Choice |
|-----------|-------------|
| Learning/teaching PCA | NIPALS (see iterations) |
| Small complete dataset | Eigenvalue (mathematically clean) |
| Large dataset | sklearn (optimized SVD) |
| Missing values | NIPALS |
| Need only first few PCs | NIPALS or sklearn |
| Production code | sklearn (tested, maintained) |

---

In [ ]:
def compare_all_methods(var_x=5.0, var_y=1.0, angle=35, n_samples=100):
    """Compare PCA results from all three algorithms."""
    # Generate data
    np.random.seed(42)  # For reproducibility
    X_comp = generate_data(n_samples, var_x, var_y, angle)
    
    # Method 1: NIPALS
    T_nipals, P_nipals, var_nipals = PCA_NIPALS(X_comp, no_components=2)
    
    # Method 2: Eigenvalue decomposition
    T_eig, P_eig, eigenvalues, _ = PCA_Eigenvalue(X_comp)
    var_eig = eigenvalues / np.sum(eigenvalues)
    
    # Method 3: sklearn
    pca_sklearn = sklearn_PCA(n_components=2)
    T_sklearn = pca_sklearn.fit_transform(X_comp)
    P_sklearn = pca_sklearn.components_.T  # Transpose to match our convention
    var_sklearn = pca_sklearn.explained_variance_ratio_
    
    # Handle sign ambiguity (eigenvectors can flip)
    for i in range(2):
        if np.dot(P_nipals[:, i], P_eig[:, i]) < 0:
            P_nipals[:, i] *= -1
            T_nipals[:, i] *= -1
        if np.dot(P_sklearn[:, i], P_eig[:, i]) < 0:
            P_sklearn[:, i] *= -1
            T_sklearn[:, i] *= -1
    
    # Create comparison figure
    fig = make_subplots(rows=2, cols=3,
                        subplot_titles=('NIPALS', 'Eigenvalue', 'sklearn',
                                       'Loadings Comparison', 'Scores Comparison', 'Variance Explained'),
                        specs=[[{'type': 'scatter'}]*3, 
                               [{'type': 'bar'}, {'type': 'scatter'}, {'type': 'bar'}]])
    
    # Row 1: Score plots
    for col, (T, name) in enumerate([(T_nipals, 'NIPALS'), (T_eig, 'Eigen'), (T_sklearn, 'sklearn')], 1):
        fig.add_trace(
            go.Scatter(x=T[:, 0], y=T[:, 1], mode='markers',
                      marker=dict(size=5, opacity=0.6), showlegend=False),
            row=1, col=col
        )
    
    # Row 2, Col 1: Loading comparison
    methods = ['NIPALS', 'Eigen', 'sklearn']
    pc1_loadings = [P_nipals[0, 0], P_eig[0, 0], P_sklearn[0, 0]]
    fig.add_trace(
        go.Bar(x=methods, y=pc1_loadings, marker=dict(color='steelblue'), showlegend=False),
        row=2, col=1
    )
    
    # Row 2, Col 2: Score correlation
    fig.add_trace(
        go.Scatter(x=T_eig[:, 0], y=T_nipals[:, 0], mode='markers',
                  marker=dict(size=5), name='NIPALS vs Eigen', showlegend=False),
        row=2, col=2
    )
    # Add diagonal
    lim = max(np.abs(T_eig[:, 0]).max(), np.abs(T_nipals[:, 0]).max())
    fig.add_trace(
        go.Scatter(x=[-lim, lim], y=[-lim, lim], mode='lines',
                  line=dict(dash='dash', color='red'), showlegend=False),
        row=2, col=2
    )
    
    # Row 2, Col 3: Variance explained
    x_labels = ['PC1', 'PC2']
    fig.add_trace(go.Bar(x=x_labels, y=var_nipals.flatten()*100, name='NIPALS', opacity=0.7), row=2, col=3)
    fig.add_trace(go.Bar(x=x_labels, y=var_eig*100, name='Eigen', opacity=0.7), row=2, col=3)
    fig.add_trace(go.Bar(x=x_labels, y=var_sklearn*100, name='sklearn', opacity=0.7), row=2, col=3)
    
    fig.update_layout(width=1200, height=800, title_text="Comparison of PCA Algorithms", barmode='group')
    fig.update_yaxes(title='PC1 Loading[0]', row=2, col=1)
    fig.update_xaxes(title='Eigenvalue PC1', row=2, col=2)
    fig.update_yaxes(title='NIPALS PC1', row=2, col=2)
    fig.update_yaxes(title='% Variance', row=2, col=3)
    fig.show()
    
    # Print numerical comparison
    print("\n" + "="*70)
    print("NUMERICAL COMPARISON")
    print("="*70)
    print(f"\nPC1 Loadings:")
    print(f"  NIPALS:  [{P_nipals[0,0]:.6f}, {P_nipals[1,0]:.6f}]")
    print(f"  Eigen:   [{P_eig[0,0]:.6f}, {P_eig[1,0]:.6f}]")
    print(f"  sklearn: [{P_sklearn[0,0]:.6f}, {P_sklearn[1,0]:.6f}]")
    
    print(f"\nVariance Explained (PC1):")
    print(f"  NIPALS:  {var_nipals.flatten()[0]*100:.2f}%")
    print(f"  Eigen:   {var_eig[0]*100:.2f}%")
    print(f"  sklearn: {var_sklearn[0]*100:.2f}%")
    
    print(f"\n✓ All methods produce equivalent results!")

# Interactive comparison
interact(
    compare_all_methods,
    var_x=FloatSlider(min=1, max=10, step=0.5, value=5, description='Var X'),
    var_y=FloatSlider(min=0.5, max=5, step=0.5, value=1, description='Var Y'),
    angle=IntSlider(min=0, max=90, step=5, value=35, description='Angle'),
    n_samples=IntSlider(min=50, max=300, step=50, value=100, description='Samples')
);

interactive(children=(FloatSlider(value=5.0, description='Var X', max=10.0, min=1.0, step=0.5), FloatSlider(va…

---

# Section 2: Compare All Three Methods

Let's verify that NIPALS, Eigenvalue decomposition, and sklearn all give the **same results**.

In [ ]:
from sklearn.decomposition import PCA
pca = PCA()  # project from 64 to 2 dimensions
score=pca.fit_transform(X)
coeff=pca.components_ #eigen vectors.T
latent = pca.explained_variance_
explained = pca.explained_variance_ratio_

In [ ]:
coeff

array([[ 1.21901390e-05,  9.99999997e-01,  7.30130279e-05],
       [-5.66460727e-01, -5.32639789e-05,  8.24088734e-01],
       [ 8.24088736e-01, -5.14047691e-05,  5.66460725e-01]])

In [ ]:
score

array([[-4.25324997e+03,  8.41288672e-01,  8.37858943e-03],
       [ 2.97275001e+03,  1.25977271e-01, -1.82476780e-01],
       [ 3.62475003e+03,  1.56843494e-01,  1.65224286e-01],
       [-2.34425007e+03, -1.12410944e+00,  8.87390454e-03]])

In [ ]:
explained

array([9.99999955e-01, 4.41567976e-08, 1.33326424e-09])

In [ ]:
latent

array([1.51872330e+07, 6.70619604e-01, 2.02485956e-02])

In [ ]:
pca.singular_values_/np.sqrt(3)

array([3.89708006e+03, 8.18913673e-01, 1.42297560e-01])

In [ ]:
np.sqrt(evals)

array([3.89708006e+03, 8.18913673e-01, 1.42297560e-01])

---

# ✅ Summary

## What You've Learned

1. **NIPALS Algorithm**
   - Iteratively finds PCs through regression
   - Can watch it converge step-by-step
   - Handles missing data (unique advantage)
   - Useful for large datasets when you only need first few PCs

2. **Eigenvalue Decomposition**
   - Direct mathematical solution via covariance matrix
   - Eigenvectors = loadings, Eigenvalues = variances
   - Mathematically elegant and interpretable
   - Best for understanding PCA theory

3. **SVD/sklearn**
   - Most numerically stable method
   - Production-ready implementation
   - Fast and well-tested
   - Best for real applications

4. **All Methods Are Equivalent**
   - Produce the same loadings (up to sign)
   - Same explained variance
   - Choose based on your specific use case

---

## 🔗 Next Steps

- [PCA_InteractiveUnderstanding.ipynb](PCA_InteractiveUnderstanding.ipynb) — Geometric intuition for projections
- [PCA_CityTemp_Plots.ipynb](PCA_CityTemp_Plots.ipynb) — Comprehensive diagnostics on real data
- [ComprehensivePCA.ipynb](ComprehensivePCA.ipynb) — Advanced validation techniques

---

**Questions?** Experiment with the interactive widgets above to build your intuition! 🎓

In [ ]:
# Standardize the planet data for comparison
X_std = (X - np.mean(X, axis=0)) / np.std(X, axis=0, ddof=1)
print("✓ Standardized planet data created")
print(f"Shape: {X_std.shape}")

In [ ]:
print("\n" + "="*70)
print("PCA RESULTS ON PLANET DATA")
print("="*70)

# NIPALS
T_n, P_n, v_n = PCA_NIPALS(X_std, 3)
print("\n✓ NIPALS completed")

# Eigenvalue
T_e, P_e, eig_vals, _ = PCA_Eigenvalue(X_std)
v_e = eig_vals / np.sum(eig_vals)
print("✓ Eigenvalue decomposition completed")

# sklearn
pca_sk = sklearn_PCA(n_components=3)
T_sk = pca_sk.fit_transform(X_std)
v_sk = pca_sk.explained_variance_ratio_
print("✓ sklearn PCA completed")

print("\n" + "-"*70)
print("Variance Explained:")
print(f"{'Method':<12} {'PC1':>10} {'PC2':>10} {'PC3':>10} {'Total':>10}")
print("-" * 52)
print(f"{'NIPALS':<12} {v_n[0][0]*100:>9.1f}% {v_n[1][0]*100:>9.1f}% {v_n[2][0]*100:>9.1f}% {(v_n[0][0]+v_n[1][0]+v_n[2][0])*100:>9.1f}%")
print(f"{'Eigenvalue':<12} {v_e[0]*100:>9.1f}% {v_e[1]*100:>9.1f}% {v_e[2]*100:>9.1f}% {np.sum(v_e)*100:>9.1f}%")
print(f"{'sklearn':<12} {v_sk[0]*100:>9.1f}% {v_sk[1]*100:>9.1f}% {v_sk[2]*100:>9.1f}% {np.sum(v_sk)*100:>9.1f}%")


PCA RESULTS ON PLANET DATA

✓ NIPALS completed
✓ Eigenvalue decomposition completed
✓ sklearn PCA completed

----------------------------------------------------------------------
Variance Explained:
Method              PC1        PC2        PC3      Total
----------------------------------------------------
NIPALS            62.3%      35.9%       1.8%     100.0%
Eigenvalue        62.3%      35.9%       1.8%     100.0%
sklearn           62.3%      35.9%       1.8%     100.0%
